In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline



In [2]:
from pathlib import Path
import pandas as pd

# Notebook is in agent1/notebooks/
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Bitext-retail-ecommerce-llm-chatbot-training-dataset.parquet"
)

print(DATA_PATH)

d:\Projects\projects\agent1\data\raw\Bitext-retail-ecommerce-llm-chatbot-training-dataset.parquet


In [3]:
df = pd.read_parquet(
    DATA_PATH,
    columns=["instruction", "intent", "category"]
)

df.head()

,instruction,intent,category
0,I got to add an item to the cart,add_product,CART
1,wanna add fucking products to the basket can h...,add_product,CART
2,i have to add products to the basket i ned help,add_product,CART
3,di like to add products to the cart could i ge...,add_product,CART
4,"I need to add an item to the cart , where do I...",add_product,CART


In [ ]:
print(df.info())

print("\nMissing values:")
print(df.isna().sum())

print("\nNumber of duplicates:")
print(df.duplicated().sum())

print("\nUnique categories:", df["category"].nunique())
print("Unique intents:", df["intent"].nunique())

In [5]:
print(
    "Duplicated instructions:",
    df["instruction"].duplicated().sum()
)

Duplicated instructions: 0


In [ ]:
category_counts = df["category"].value_counts()
intent_counts = df["intent"].value_counts()

display(category_counts)
display(intent_counts)

In [7]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["intent"]
)

In [10]:
X_train = train_df["instruction"]
X_test  = test_df["instruction"]

y_intent_train = train_df["intent"]
y_intent_test  = test_df["intent"]

y_category_train = train_df["category"]
y_category_test  = test_df["category"]

In [8]:
intent_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    )
])

In [11]:
intent_model.fit(X_train, y_intent_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](46,)","['add_product','availability','availability_in_store',...,'track_order', 'use_app','wrong_item']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'


In [ ]:
#evaluation
from sklearn.metrics import accuracy_score, classification_report

y_pred_intent = intent_model.predict(X_test)

print(
    "Intent accuracy:",
    accuracy_score(y_intent_test, y_pred_intent)
)

print(
    classification_report(
        y_intent_test,
        y_pred_intent
    )
)

In [13]:
#category model
category_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    )
])

category_model.fit(X_train, y_category_train)

y_pred_category = category_model.predict(X_test)

print(
    "Category accuracy:",
    accuracy_score(
        y_category_test,
        y_pred_category
    )
)

Category accuracy: 0.9923136905424975


In [ ]:
#implementation of the mixed model: intent plus category
intent_to_category = (
    df[["intent", "category"]]
    .drop_duplicates()
    .set_index("intent")["category"]
    .to_dict()
)

intent_to_category

In [17]:
import numpy as np

intent_probs = intent_model.predict_proba(X_test)
category_probs = category_model.predict_proba(X_test)

intent_classes = intent_model.named_steps["classifier"].classes_
category_classes = category_model.named_steps["classifier"].classes_

In [18]:
category_idx = {
    category: idx
    for idx, category in enumerate(category_classes)
}

In [19]:
#soft hierarchical scores
alpha = 1.0
combined_scores = intent_probs.copy()

for j, intent in enumerate(intent_classes):
    category = intent_to_category[intent]
    c_idx = category_idx[category]

    combined_scores[:, j] *= (
        category_probs[:, c_idx] ** alpha
    )

In [20]:
combined_pred = intent_classes[
    np.argmax(combined_scores, axis=1)
]

In [21]:
# Evaluation of the combined model
hierarchical_accuracy = accuracy_score(
    y_intent_test,
    combined_pred
)

print(f"Flat intent accuracy: {accuracy_score(y_intent_test, y_pred_intent):.4f}")
print(f"Soft hierarchical accuracy: {hierarchical_accuracy:.4f}")

Flat intent accuracy: 0.9862
Soft hierarchical accuracy: 0.9864


In [23]:
#Second method: semantic plus regression model
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [24]:
X_train_emb = embedding_model.encode(
    X_train.tolist(),
    show_progress_bar=True
)

X_test_emb = embedding_model.encode(
    X_test.tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/1123 [00:00<?, ?it/s]

Batches:   0%|          | 0/281 [00:00<?, ?it/s]

In [25]:
#savings
EMBEDDINGS_DIR = PROJECT_ROOT / "data" / "processed" / "embeddings"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

np.save(EMBEDDINGS_DIR / "X_train_embeddings.npy", X_train_emb)
np.save(EMBEDDINGS_DIR / "X_test_embeddings.npy", X_test_emb)

In [ ]:
#Loading embeddings whenever we need to use them again, without recomputing:
X_train_emb = np.load(
    EMBEDDINGS_DIR / "X_train_embeddings.npy"
)

X_test_emb = np.load(
    EMBEDDINGS_DIR / "X_test_embeddings.npy"
)

In [26]:
semantic_intent_model = LogisticRegression(
    max_iter=1000
)

semantic_intent_model.fit(
    X_train_emb,
    y_intent_train
)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [27]:


y_pred_semantic = semantic_intent_model.predict(
    X_test_emb
)

semantic_accuracy = accuracy_score(
    y_intent_test,
    y_pred_semantic
)

print(f"Semantic intent accuracy: {semantic_accuracy:.4f}")

Semantic intent accuracy: 0.9882


In [28]:
#Comparing the two models
tfidf_pred = intent_model.predict(X_test)

semantic_pred = semantic_intent_model.predict(X_test_emb)

comparison = pd.DataFrame({
    "instruction": X_test.values,
    "true_intent": y_intent_test.values,
    "tfidf_pred": tfidf_pred,
    "semantic_pred": semantic_pred
})

In [ ]:
#cases where semantic model fixed the tfidf model's mistakes
semantic_fixes = comparison[
    (comparison["tfidf_pred"] != comparison["true_intent"]) &
    (comparison["semantic_pred"] == comparison["true_intent"])
]

print(len(semantic_fixes))
display(semantic_fixes.head(20))

In [ ]:
#cases where tfidf model fixed the semantic model's mistakes
tfidf_fixes = comparison[
    (comparison["tfidf_pred"] == comparison["true_intent"]) &
    (comparison["semantic_pred"] != comparison["true_intent"])
]

print(len(tfidf_fixes))
display(tfidf_fixes.head(20))

In [ ]:
#both models made the same mistake
both_wrong = comparison[
    (comparison["tfidf_pred"] != comparison["true_intent"]) &
    (comparison["semantic_pred"] != comparison["true_intent"])
]

print(len(both_wrong))
display(both_wrong.head(20))

In [32]:
print("Semantic fixes TF-IDF:", len(semantic_fixes))
print("TF-IDF fixes semantic:", len(tfidf_fixes))
print("Both wrong:", len(both_wrong))

Semantic fixes TF-IDF: 74
TF-IDF fixes semantic: 56
Both wrong: 50


In [33]:
#Ensemble

tfidf_probs = intent_model.predict_proba(X_test)
semantic_probs = semantic_intent_model.predict_proba(X_test_emb)

tfidf_classes = intent_model.named_steps["classifier"].classes_
semantic_classes = semantic_intent_model.classes_

assert np.array_equal(tfidf_classes, semantic_classes)

ensemble_probs = (
    0.5 * tfidf_probs +
    0.5 * semantic_probs
)

ensemble_pred = semantic_classes[
    np.argmax(ensemble_probs, axis=1)
]

ensemble_accuracy = accuracy_score(
    y_intent_test,
    ensemble_pred
)

print(f"TF-IDF:   {accuracy_score(y_intent_test, tfidf_pred):.4f}")
print(f"Semantic: {accuracy_score(y_intent_test, semantic_pred):.4f}")
print(f"Ensemble: {ensemble_accuracy:.4f}")

TF-IDF:   0.9862
Semantic: 0.9882
Ensemble: 0.9910
